In [21]:
import pandas as pd
import numpy as np
import itertools
import os
from datetime import date, timedelta
from tqdm import tqdm
from utils import scenario_name



In [22]:
import os
from datetime import date, timedelta
import yaml

# ─── LOAD CONFIG ────────────────────────────────────────────────────────────────
with open(os.path.join(os.getcwd(), "config.yaml"), "r") as f:
    cfg = yaml.safe_load(f)

DATA_PATH        = cfg["data"]["flattened_measurements"]
OUT_DIR          = cfg["output"]["scenario_features"]
os.makedirs(OUT_DIR, exist_ok=True)

PROPERTIES       = cfg["scenario"]["properties"]

# Floors: static list or dynamic from data
if cfg["scenario"].get("dynamic", False):
    # will fill in after df load
    FLOORS = None
else:
    FLOORS = cfg["scenario"]["floors"]["static"]

# Time window
week_start_str   = cfg["window"]["week_start"]
WEEK_START       = date.fromisoformat(week_start_str)
DAYS             = [WEEK_START + timedelta(days=i) for i in range(cfg["window"]["days"])]

# ─── LOAD DATA ─────────────────────────────────────────────────────────────────
import pandas as pd
df = pd.read_parquet(DATA_PATH)

# Optionally override floors dynamically:
if FLOORS is None:
    unique_floors = sorted(df["floor"].dropna().unique().tolist())
    FLOORS = [[f] for f in unique_floors]

# ─── BUILD SCENARIOS ────────────────────────────────────────────────────────────
import itertools
from utils import scenario_name

SCENARIO_LIST = list(itertools.product(PROPERTIES, FLOORS))
print(f"Total scenarios to generate: {len(SCENARIO_LIST)}")


Total scenarios to generate: 28


In [23]:
# Coerce 'floor' to numeric, set errors='coerce' to turn any non-numeric into NaN (should be rare)
df['floor'] = pd.to_numeric(df['floor'], errors='coerce')

# Fill NaN floors with 0 (unknown/ambiguous), then cast to int
df['floor'] = df['floor'].fillna(0).astype(int)

# For debugging: check available properties and floors
print("Available properties:", df['property'].unique()[:20])
print("Available floors:", sorted(df['floor'].unique()))

# Make sure 'property' is string for filtering
df['property'] = df['property'].astype(str)


Available properties: ['https://interconnectproject.eu/example/property_R5_15__temp_'
 'https://interconnectproject.eu/example/property_R5_15__co2_'
 'https://interconnectproject.eu/example/property_R5_15__humidity_'
 'https://interconnectproject.eu/example/property_R5_2__temp_'
 'https://interconnectproject.eu/example/property_R5_2__humidity_'
 'https://interconnectproject.eu/example/property_R5_2__co2_'
 'https://interconnectproject.eu/example/property_R5_23__temp_'
 'https://interconnectproject.eu/example/property_R5_23__co2_'
 'https://interconnectproject.eu/example/property_R5_23__humidity_'
 'https://interconnectproject.eu/example/property_R5_3__humidity_'
 'https://interconnectproject.eu/example/property_R5_3__temp_'
 'https://interconnectproject.eu/example/property_R5_3__co2_'
 'https://interconnectproject.eu/example/property_R5_44__temp_'
 'https://interconnectproject.eu/example/property_R5_44__humidity_'
 'https://interconnectproject.eu/example/property_R5_44__co2_'
 'https:/

In [24]:


total_saved = 0
skipped_no_data = []
skipped_too_few = []
skipped_empty = []
for props, floors in tqdm(SCENARIO_LIST, desc="Scenarios"):
    # Filter to sensors matching ANY of the property mashup (case-insensitive substring match)
    prop_mask = df['property'].apply(lambda x: any(p.lower() in x.lower() for p in props))
    # Filter to sensors on ANY of the selected floors (excluding floor 0)
    floor_mask = df['floor'].isin(floors) & (df['floor'] != 0)
    df_scenario = df[prop_mask & floor_mask].copy()
    
    scenario_id = scenario_name(props, floors)
    if df_scenario.empty:
        skipped_no_data.append(scenario_id)
        continue

    # Add 'day' column for grouping
    df_scenario['day'] = df_scenario['timestamp'].dt.date

    # Only include target week days
    df_scenario = df_scenario[df_scenario['day'].isin(DAYS)]

    # For each day in week, compute features per sensor
    for day in DAYS:
        df_day = df_scenario[df_scenario['day'] == day]
        if df_day.empty:
            skipped_empty.append((scenario_id, day, 'no data for day'))

            continue

      
        features = df_day.groupby(['device', 'property']).agg(
            mean=('value', 'mean'),
            std=('value', 'std'),
            min=('value', 'min'),
            max=('value', 'max'),
            count=('value', 'count')
        ).reset_index()
        features['day'] = day

        # Only save if there are at least 10 sensors (for clustering validity)
        if len(features) < 10:
            skipped_too_few.append((scenario_id, day, f"only {len(features)} sensors"))
            continue

        # Save with scenario and day in filename
        fname = f"{OUT_DIR}/{scenario_name(props, floors)}__{day}.csv"
        features.to_csv(fname, index=False)
        total_saved += 1


print(f"Total scenario-day feature matrices saved: {total_saved}")


Scenarios: 100%|██████████| 28/28 [04:16<00:00,  9.14s/it]

Total scenario-day feature matrices saved: 196


In [25]:
print("Skipped scenarios with no data:", len(skipped_no_data))
print("Skipped scenarios with too few sensors:", len(skipped_too_few))
print("Skipped empty days:", len(skipped_empty))
skipped_too_few

Skipped scenarios with no data: 0
Skipped scenarios with too few sensors: 0
Skipped empty days: 0


[]

In [27]:
import pandas as pd
from pathlib import Path

# Locate the specific scenario CSV file in the directory
dir_path = Path("scenario_features_batch")
files = list(dir_path.glob("CO2_Humidity_temp__floors1_2_3_4_5_6_7__2022-03-08.csv"))
if not files:
    raise FileNotFoundError("Specified scenario file not found in scenario_features_batch directory.")
file_path = files[0]

# Load the CSV
df = pd.read_csv(file_path)

files = list(dir_path.glob("temp__floors1_2_3_4_5_6_7__2022-03-08.csv"))
if not files:
    raise FileNotFoundError("Specified scenario file not found in scenario_features_batch directory.")
file_path = files[0]
df2 = pd.read_csv(file_path)
# Extract measurement type from 'property' column
def extract_measurement_type(prop_str):
    prop_str = prop_str.lower()
    if "temp" in prop_str:
        return "temperature"
    if "co2" in prop_str:
        return "co2"
    if "humidity" in prop_str:
        return "humidity"
    return "other"

df["measurement_type"] = df["property"].apply(extract_measurement_type)
df2["measurement_type"] = df2["property"].apply(extract_measurement_type)

# Count how many entries for each measurement type
counts = df["measurement_type"].value_counts().reset_index()
counts.columns = ["measurement_type", "count"]

counts2 = df2["measurement_type"].value_counts().reset_index()
counts2.columns = ["measurement_type", "count"]
# Display results to the user
print("Measurement Count by Type forCO2_Humidity_Temperature__floors1_2_3_4_5_6_7__2022 Scenario", counts)
print("Measurement Count by Type for temp__floors1_2_3_4_5_6_7__2022 Scenario", counts2)


Measurement Count by Type forCO2_Humidity_Temperature__floors1_2_3_4_5_6_7__2022 Scenario   measurement_type  count
0      temperature    280
1              co2    222
2         humidity    222
Measurement Count by Type for temp__floors1_2_3_4_5_6_7__2022 Scenario   measurement_type  count
0      temperature    280
